# WM-04 · Dreamer 风格 RSSM 小世界模型（自包含）

完整 DreamerV3 长训 Atari 在 Kaggle 会话里不现实。这里实现 **RSSM 核心**：

- 确定性 h（GRU）+ 随机 z  
- 用重建像素训练  
- 在 latent 里 dream 出轨迹  

同样用弹球玩具环境，保证可跑通，并理解 Dreamer 骨架。

In [ ]:
import math, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
import imageio.v2 as imageio

assert torch.cuda.is_available()
device=torch.device('cuda')
print(torch.cuda.get_device_name(0))
OUT=Path('/kaggle/working/wm_rssm'); OUT.mkdir(exist_ok=True)

class BallEnv:
    def __init__(self, size=48):
        self.size=size; self.reset()
    def reset(self):
        self.p=np.array([random.uniform(0.25,0.75), random.uniform(0.25,0.75)])
        ang=random.uniform(0,2*math.pi); sp=random.uniform(0.04,0.07)
        self.v=np.array([math.cos(ang), math.sin(ang)])*sp
        return self.render()
    def step(self, a=0):
        force=np.array([[0,0],[-1,0],[1,0],[0,-1],[0,1]][a])*0.012
        self.v = self.v + force
        self.p = self.p + self.v
        for i in range(2):
            if self.p[i]<0.08 or self.p[i]>0.92:
                self.v[i]*=-1
                self.p[i]=float(np.clip(self.p[i],0.08,0.92))
        return self.render()
    def render(self):
        s=self.size
        img=Image.new('RGB',(s,s),(12,14,22)); d=ImageDraw.Draw(img)
        x,y=self.p*s; d.ellipse([x-5,y-5,x+5,y+5], fill=(255,180,60))
        return np.asarray(img).astype(np.float32)/255.

def episodes(n=30,T=30):
    env=BallEnv(); xs=[]; as_=[]
    for _ in range(n):
        o=env.reset(); xs.append(o)
        for t in range(T-1):
            a=random.randint(0,4); o=env.step(a); xs.append(o); as_.append(a)
        as_.append(0)
    x=torch.tensor(np.stack(xs)).permute(0,3,1,2).float()
    a=torch.tensor(as_, dtype=torch.long)
    return x,a

X,A=episodes()
print(X.shape)

In [ ]:
# Minimal RSSM
class ObsEncoder(nn.Module):
    def __init__(self, out=64):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,64,4,2,1), nn.ReLU(),
            nn.Flatten(), nn.Linear(64*6*6, out), nn.ReLU(),
        )
    def forward(self,x): return self.net(x)

class ObsDecoder(nn.Module):
    def __init__(self, inn=64+32):
        super().__init__()
        self.fc=nn.Linear(inn, 64*6*6)
        self.net=nn.Sequential(
            nn.ConvTranspose2d(64,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1), nn.Sigmoid(),
        )
    def forward(self, hz):
        h=self.fc(hz).view(-1,64,6,6); return self.net(h)

class RSSM(nn.Module):
    def __init__(self, stoch=32, deter=64, act=5):
        super().__init__()
        self.stoch=stoch; self.deter=deter
        self.embed_a=nn.Embedding(act, 16)
        self.gru=nn.GRUCell(stoch+16, deter)
        self.prior=nn.Linear(deter, 2*stoch)
        self.post=nn.Linear(deter+64, 2*stoch)
        self.enc=ObsEncoder(64)
        self.dec=ObsDecoder(deter+stoch)
    def split(self, stats):
        mu,lv=stats.chunk(2,-1); std=F.softplus(lv)+1e-3
        return mu, std
    def forward_seq(self, obs, actions):
        # obs B,T,C,H,W  actions B,T
        B,T=actions.shape
        h=torch.zeros(B, self.deter, device=obs.device)
        z=torch.zeros(B, self.stoch, device=obs.device)
        recons=[]; kl_sum=0; posts=[]
        for t in range(T):
            a=self.embed_a(actions[:,t])
            h=self.gru(torch.cat([z,a],-1), h)
            prior=self.split(self.prior(h))
            e=self.enc(obs[:,t])
            post=self.split(self.post(torch.cat([h,e],-1)))
            mu,std=post
            z=mu+std*torch.randn_like(std)
            rec=self.dec(torch.cat([h,z],-1))
            recons.append(rec)
            # KL post||prior
            pmu,pstd=prior
            kl=0.5*torch.mean((std**2+ (mu-pmu)**2)/(pstd**2+1e-8) -1 + 2*torch.log(pstd/(std+1e-8)))
            kl_sum=kl_sum+kl
            posts.append(z)
        return torch.stack(recons,1), kl_sum/T

# sequences
T=16
seq_x=[]; seq_a=[]
for i in range(0, len(X)-T, T):
    seq_x.append(X[i:i+T]); seq_a.append(A[i:i+T])
seq_x=torch.stack(seq_x); seq_a=torch.stack(seq_a)
print('seq', seq_x.shape)

rssm=RSSM().to(device)
opt=torch.optim.Adam(rssm.parameters(), lr=2e-3)
for ep in range(60):
    rssm.train()
    idx=torch.randperm(len(seq_x))[:16]
    obs=seq_x[idx].to(device); act=seq_a[idx].to(device)
    rec,kl=rssm.forward_seq(obs, act)
    loss=F.mse_loss(rec, obs)+0.1*kl
    opt.zero_grad(); loss.backward(); opt.step()
    if ep%10==0: print(f'ep {ep} loss={loss.item():.4f} kl={float(kl):.4f}')

In [ ]:
# Dream open-loop from prior
rssm.eval()
with torch.no_grad():
    obs0=seq_x[0,:1].to(device)  # 1,T? take first frame of seq0
    # encode first step posterior then roll prior only
    B=1; h=torch.zeros(B,rssm.deter,device=device); z=torch.zeros(B,rssm.stoch,device=device)
    frames=[]
    o0=seq_x[0,0:1].to(device)
    a0=seq_a[0,0:1].to(device)
    a_emb=rssm.embed_a(a0)
    h=rssm.gru(torch.cat([z,a_emb],-1), h)
    e=rssm.enc(o0)
    mu,std=rssm.split(rssm.post(torch.cat([h,e],-1)))
    z=mu
    frames.append(rssm.dec(torch.cat([h,z],-1))[0].permute(1,2,0).cpu().numpy())
    for t in range(40):
        a=torch.randint(0,5,(1,),device=device)
        a_emb=rssm.embed_a(a)
        h=rssm.gru(torch.cat([z,a_emb],-1), h)
        mu,std=rssm.split(rssm.prior(h))
        z=mu  # mean dream
        img=rssm.dec(torch.cat([h,z],-1))[0].permute(1,2,0).cpu().numpy()
        frames.append(img)

dream=[(f*255).clip(0,255).astype(np.uint8) for f in frames]
real=[(seq_x[0,t].permute(1,2,0).numpy()*255).astype(np.uint8) for t in range(min(40,seq_x.shape[1]))]
imageio.mimsave(OUT/'rssm_dream.gif', dream, fps=10, loop=0)
imageio.mimsave(OUT/'rssm_real.gif', real, fps=10, loop=0)
# strip
from PIL import Image as PImage
def strip(fs,n=8):
    idxs=np.linspace(0,len(fs)-1,n,dtype=int)
    ims=[PImage.fromarray(fs[i]).resize((48,48)) for i in idxs]
    s=PImage.new('RGB',(48*n,48))
    for i,im in enumerate(ims): s.paste(im,(i*48,0))
    return s
both=PImage.new('RGB',(48*8, 48*2+6),(0,0,0))
both.paste(strip(real),(0,0)); both.paste(strip(dream),(0,54))
both.save(OUT/'rssm_real_vs_dream.png')
print('saved', list(OUT.iterdir()))
try:
    from IPython.display import display, Image as IImage
    display(IImage(filename=str(OUT/'rssm_dream.gif')))
    display(both)
except Exception: pass
import shutil
shutil.make_archive('/kaggle/working/wm_rssm_export','zip', OUT)
print('04 RSSM DONE')